In [ ]:
## Added PSIR and Time inference metrics along with existing Evaluation parameter

In [ ]:
from pathlib import Path
import time
import json
import shutil
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

import cv2
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from skimage.metrics import peak_signal_noise_ratio, structural_similarity

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

PROCESSED_DIR = Path("../data/processed")
CHECKPOINT_PATH = Path("../outputs/checkpoints/best_physics_unet_hybrid.pth")

OUTPUT_DIR = Path("../outputs/final_enhancement")
PRED_DIR = OUTPUT_DIR / "predicted"
GT_DIR = OUTPUT_DIR / "ground_truth"
ENHANCED_DIR = OUTPUT_DIR / "enhanced_sr"

for d in [PRED_DIR, GT_DIR, ENHANCED_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Device:", device)
print("Checkpoint exists:", CHECKPOINT_PATH.exists())

In [ ]:
class LandsatPatchDataset(Dataset):
    def __init__(self, processed_dir, split):
        self.input_dir = Path(processed_dir) / split / "inputs"
        self.target_dir = Path(processed_dir) / split / "targets"

        self.input_files = sorted(self.input_dir.glob("*.npy"))
        self.target_files = sorted(self.target_dir.glob("*.npy"))

        assert len(self.input_files) == len(self.target_files)

    def __len__(self):
        return len(self.input_files)

    def __getitem__(self, idx):
        inp = np.load(self.input_files[idx]).astype(np.float32)
        tgt = np.load(self.target_files[idx]).astype(np.float32)

        inp = torch.from_numpy(inp).permute(2, 0, 1)
        tgt = torch.from_numpy(tgt).permute(2, 0, 1)

        return inp, tgt

In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()

        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),

            nn.Conv2d(out_channels, out_channels, 3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)


class PhysicsUNet(nn.Module):
    def __init__(self, in_channels=5, out_channels=3):
        super().__init__()

        self.down1 = DoubleConv(in_channels, 32)
        self.pool1 = nn.MaxPool2d(2)

        self.down2 = DoubleConv(32, 64)
        self.pool2 = nn.MaxPool2d(2)

        self.down3 = DoubleConv(64, 128)
        self.pool3 = nn.MaxPool2d(2)

        self.down4 = DoubleConv(128, 256)
        self.pool4 = nn.MaxPool2d(2)

        self.bottleneck = DoubleConv(256, 512)

        self.up4 = nn.ConvTranspose2d(512, 256, 2, 2)
        self.conv4 = DoubleConv(512, 256)

        self.up3 = nn.ConvTranspose2d(256, 128, 2, 2)
        self.conv3 = DoubleConv(256, 128)

        self.up2 = nn.ConvTranspose2d(128, 64, 2, 2)
        self.conv2 = DoubleConv(128, 64)

        self.up1 = nn.ConvTranspose2d(64, 32, 2, 2)
        self.conv1 = DoubleConv(64, 32)

        self.out = nn.Conv2d(32, out_channels, 1)
        self.activation = nn.Sigmoid()

    def forward(self, x):
        d1 = self.down1(x)
        d2 = self.down2(self.pool1(d1))
        d3 = self.down3(self.pool2(d2))
        d4 = self.down4(self.pool3(d3))

        bn = self.bottleneck(self.pool4(d4))

        u4 = self.up4(bn)
        u4 = self.conv4(torch.cat([u4, d4], dim=1))

        u3 = self.up3(u4)
        u3 = self.conv3(torch.cat([u3, d3], dim=1))

        u2 = self.up2(u3)
        u2 = self.conv2(torch.cat([u2, d2], dim=1))

        u1 = self.up1(u2)
        u1 = self.conv1(torch.cat([u1, d1], dim=1))

        return self.activation(self.out(u1))

In [ ]:
model = PhysicsUNet(in_channels=5, out_channels=3).to(device)

checkpoint = torch.load(CHECKPOINT_PATH, map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

print("Loaded:", checkpoint.get("model_name", "PhysicsUNet_HybridLoss"))
print("Epoch:", checkpoint["epoch"] + 1)
print("Best Val Loss:", checkpoint["val_loss"])

In [ ]:
test_dataset = LandsatPatchDataset(PROCESSED_DIR, "test")

test_loader = DataLoader(
    test_dataset,
    batch_size=1,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

print("Test samples:", len(test_dataset))

In [ ]:
def to_uint8(img):
    img = np.clip(img, 0, 1)
    return (img * 255).astype(np.uint8)


def enhance_sr_rgb(img, scale=2):
    """
    Lightweight enhancement module:
    1. Bicubic 2x upscaling
    2. CLAHE on LAB luminance
    3. Unsharp masking
    """

    img_uint8 = to_uint8(img)

    h, w, _ = img_uint8.shape

    # 1. Bicubic Super-Resolution style upscaling
    upscaled = cv2.resize(
        img_uint8,
        (w * scale, h * scale),
        interpolation=cv2.INTER_CUBIC
    )

    # 2. CLAHE contrast enhancement in LAB color space
    lab = cv2.cvtColor(upscaled, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)

    clahe = cv2.createCLAHE(
        clipLimit=2.0,
        tileGridSize=(8, 8)
    )

    l_enhanced = clahe.apply(l)

    lab_enhanced = cv2.merge([l_enhanced, a, b])
    enhanced = cv2.cvtColor(lab_enhanced, cv2.COLOR_LAB2RGB)

    # 3. Unsharp masking
    blurred = cv2.GaussianBlur(enhanced, (0, 0), sigmaX=1.0)
    sharpened = cv2.addWeighted(enhanced, 1.5, blurred, -0.5, 0)

    return np.clip(sharpened, 0, 255).astype(np.uint8)

In [ ]:
# Clean old images
for folder in [PRED_DIR, GT_DIR, ENHANCED_DIR]:
    for f in folder.glob("*.png"):
        f.unlink()

psnr_scores = []
ssim_scores = []
inference_times = []
enhancement_times = []

MAX_IMAGES = 500  # for FID, 500 is enough and faster. Increase if needed.

with torch.no_grad():
    for idx, (inp, tgt) in enumerate(tqdm(test_loader)):

        if idx >= MAX_IMAGES:
            break

        inp = inp.to(device, non_blocking=True)

        if torch.cuda.is_available():
            torch.cuda.synchronize()

        start = time.time()
        pred = model(inp)

        if torch.cuda.is_available():
            torch.cuda.synchronize()

        end = time.time()

        inference_times.append((end - start) * 1000)

        pred_np = pred.squeeze(0).cpu().permute(1, 2, 0).numpy()
        tgt_np = tgt.squeeze(0).permute(1, 2, 0).numpy()

        pred_np = np.clip(pred_np, 0, 1)
        tgt_np = np.clip(tgt_np, 0, 1)

        psnr_scores.append(
            peak_signal_noise_ratio(tgt_np, pred_np, data_range=1.0)
        )

        ssim_scores.append(
            structural_similarity(
                tgt_np,
                pred_np,
                channel_axis=-1,
                data_range=1.0
            )
        )

        enh_start = time.time()
        enhanced_img = enhance_sr_rgb(pred_np, scale=2)
        enh_end = time.time()

        enhancement_times.append((enh_end - enh_start) * 1000)

        pred_img = to_uint8(pred_np)
        gt_img = to_uint8(tgt_np)

        plt.imsave(PRED_DIR / f"pred_{idx:04d}.png", pred_img)
        plt.imsave(GT_DIR / f"gt_{idx:04d}.png", gt_img)
        plt.imsave(ENHANCED_DIR / f"enhanced_{idx:04d}.png", enhanced_img)

print("Saved predictions:", len(list(PRED_DIR.glob('*.png'))))
print("Saved ground truth:", len(list(GT_DIR.glob('*.png'))))
print("Saved enhanced:", len(list(ENHANCED_DIR.glob('*.png'))))

In [ ]:
avg_psnr = float(np.mean(psnr_scores))
avg_ssim = float(np.mean(ssim_scores))
avg_inference_ms = float(np.mean(inference_times))
avg_enhancement_ms = float(np.mean(enhancement_times))
avg_total_ms = avg_inference_ms + avg_enhancement_ms
fps = 1000.0 / avg_total_ms

print("=" * 60)
print("FINAL MODEL METRICS")
print("=" * 60)
print(f"PSNR: {avg_psnr:.4f}")
print(f"SSIM: {avg_ssim:.4f}")
print(f"Average Model Inference Time: {avg_inference_ms:.3f} ms/tile")
print(f"Average SR Enhancement Time: {avg_enhancement_ms:.3f} ms/tile")
print(f"Average Total Pipeline Time: {avg_total_ms:.3f} ms/tile")
print(f"Approx FPS: {fps:.2f}")
print("=" * 60)

In [ ]:
def show_final_comparison(idx=None):
    if idx is None:
        idx = np.random.randint(0, min(MAX_IMAGES, len(test_dataset)))

    pred_img = plt.imread(PRED_DIR / f"pred_{idx:04d}.png")
    gt_img = plt.imread(GT_DIR / f"gt_{idx:04d}.png")
    enhanced_img = plt.imread(ENHANCED_DIR / f"enhanced_{idx:04d}.png")

    plt.figure(figsize=(18, 6))

    plt.subplot(1, 3, 1)
    plt.imshow(pred_img)
    plt.title("Model Output RGB")
    plt.axis("off")

    plt.subplot(1, 3, 2)
    plt.imshow(enhanced_img)
    plt.title("SR + Sharpened Output")
    plt.axis("off")

    plt.subplot(1, 3, 3)
    plt.imshow(gt_img)
    plt.title("Ground Truth RGB")
    plt.axis("off")

    plt.tight_layout()
    plt.show()


for _ in range(5):
    show_final_comparison()

In [ ]:
from torch_fidelity import calculate_metrics

fid_metrics = calculate_metrics(
    input1=str(PRED_DIR),
    input2=str(GT_DIR),
    cuda=torch.cuda.is_available(),
    fid=True,
    isc=False,
    kid=False,
    verbose=True
)

fid_score = float(fid_metrics["frechet_inception_distance"])

print("FID Score:", fid_score)

In [ ]:
fid_enhanced_metrics = calculate_metrics(
    input1=str(ENHANCED_DIR),
    input2=str(GT_DIR),
    cuda=torch.cuda.is_available(),
    fid=True,
    isc=False,
    kid=False,
    verbose=True
)

fid_enhanced_score = float(fid_enhanced_metrics["frechet_inception_distance"])

print("FID Score - Raw Prediction:", fid_score)
print("FID Score - SR Enhanced:", fid_enhanced_score)

In [ ]:
metrics = {
    "model": "PhysicsUNet_HybridLoss",
    "checkpoint": str(CHECKPOINT_PATH),
    "samples_used": MAX_IMAGES,
    "psnr": avg_psnr,
    "ssim": avg_ssim,
    "fid_raw_prediction": fid_score,
    "fid_sr_enhanced": fid_enhanced_score,
    "avg_model_inference_ms_per_tile": avg_inference_ms,
    "avg_sr_enhancement_ms_per_tile": avg_enhancement_ms,
    "avg_total_pipeline_ms_per_tile": avg_total_ms,
    "fps": fps,
    "sr_module": "Bicubic 2x + CLAHE + Unsharp Masking"
}

METRICS_DIR = Path("../outputs/metrics")
METRICS_DIR.mkdir(parents=True, exist_ok=True)

with open(METRICS_DIR / "final_model_metrics_sr_fid_inference.json", "w") as f:
    json.dump(metrics, f, indent=4)

print("Saved:", METRICS_DIR / "final_model_metrics_sr_fid_inference.json")